# Evaluation, Adapter Merging, and Serving

Fine-tuning is not finished when the loss goes down. You need to evaluate behavior, compare against the base model, save artifacts clearly, decide whether to merge adapters, and prepare a serving path.

This notebook closes the Week 5-6 loop: train, evaluate, package, serve.

## What You Will Learn

By the end, you should understand:

- why training loss is not enough,
- how to build a small regression eval set,
- simple automatic metrics for narrow tasks,
- qualitative side-by-side comparison,
- how PEFT adapters are loaded and merged,
- when not to merge adapters,
- how to prepare artifacts for vLLM or Transformers serving,
- and what belongs in a fine-tuning final report.

## 1. Evaluation Mindset

Fine-tuning can improve one behavior while damaging another. Evaluate at multiple levels:

| Eval type | Use | Example |
|---|---|---|
| Loss / perplexity | language modeling signal | Did held-out loss improve? |
| Task metric | objective task score | Exact match, F1, accuracy |
| Behavioral rubric | human-readable quality | Helpfulness, correctness, style |
| Regression set | protect critical behavior | Does the model still refuse unsafe requests? |
| Side-by-side comparison | compare base vs tuned | Which answer is better and why? |
| Production metric | serving quality and cost | latency, tokens/sec, error rate |

In [ ]:
import importlib.util
from pathlib import Path

import pandas as pd

def has_package(name):
    return importlib.util.find_spec(name) is not None

packages = ["torch", "transformers", "peft", "evaluate"]
pd.DataFrame({"package": packages, "installed": [has_package(name) for name in packages]})

## 2. A Tiny Regression Eval Set

A regression set is a small collection of prompts you care about. Keep it stable across runs so you can compare model versions. For serious projects, this should include normal tasks, hard tasks, edge cases, refusal boundaries, and examples from real users.

In [ ]:
eval_records = [
    {
        "id": "lora-basic",
        "prompt": "Explain LoRA in two sentences.",
        "must_contain": ["low-rank", "adapter"],
    },
    {
        "id": "qlora-memory",
        "prompt": "What problem does QLoRA solve?",
        "must_contain": ["4-bit", "memory"],
    },
    {
        "id": "rag-vs-ft",
        "prompt": "When should I use RAG instead of fine-tuning?",
        "must_contain": ["external", "facts"],
    },
]
pd.DataFrame(eval_records)

## 3. Simple String-Based Metrics

String checks are not enough for open-ended LLM quality, but they are useful as quick regression tests for narrow expectations. They catch obvious failures and are cheap to run after every training attempt.

In [ ]:
sample_outputs = {
    "lora-basic": "LoRA is a low-rank adapter method that freezes the base model and trains small update matrices. It is parameter-efficient.",
    "qlora-memory": "QLoRA reduces memory by loading the base model in 4-bit precision while training LoRA adapters.",
    "rag-vs-ft": "Use RAG when the answer depends on external facts or frequently changing documents rather than a learned behavior.",
}

def score_must_contain(output, required_terms):
    normalized = output.lower()
    hits = [term for term in required_terms if term.lower() in normalized]
    return len(hits) / max(1, len(required_terms)), hits

rows = []
for record in eval_records:
    output = sample_outputs[record["id"]]
    score, hits = score_must_contain(output, record["must_contain"])
    rows.append({
        "id": record["id"],
        "score": score,
        "hits": hits,
        "output": output,
    })
pd.DataFrame(rows)

## 4. Side-by-Side Evaluation Template

For open-ended responses, compare base and tuned outputs with a rubric. The rubric should reflect your use case, not generic vibes.

In [ ]:
comparison_template = pd.DataFrame([
    {
        "prompt_id": "lora-basic",
        "criterion": "technical correctness",
        "base_score_1_to_5": None,
        "tuned_score_1_to_5": None,
        "winner": None,
        "notes": None,
    },
    {
        "prompt_id": "lora-basic",
        "criterion": "conciseness",
        "base_score_1_to_5": None,
        "tuned_score_1_to_5": None,
        "winner": None,
        "notes": None,
    },
])
comparison_template

## 5. Load a Base Model and Adapter

PEFT adapters are usually saved separately from the base model. To reproduce a fine-tuned model, you need both the base model ID and the adapter directory. This cell is guarded because it depends on packages and saved adapters from earlier notebooks.

In [ ]:
base_model_name = "sshleifer/tiny-gpt2"
adapter_dir = Path("finetuning_outputs/lora_sft/adapter")

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir if adapter_dir.exists() else base_model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    if adapter_dir.exists():
        tuned_model = PeftModel.from_pretrained(base_model, adapter_dir)
        print("loaded adapter")
    else:
        tuned_model = None
        print("adapter directory does not exist yet; run the PEFT LoRA notebook first")
except Exception as error:
    tokenizer = None
    base_model = None
    tuned_model = None
    print("Adapter load skipped. Install transformers and peft first.")
    print(f"Reason: {error}")

## 6. Generate Responses for Comparison

For real evaluation, generate base and tuned outputs with the same decoding settings. Keep temperature low for deterministic regression tests and higher only when evaluating creative behavior.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=80):
    import torch

    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

try:
    prompt = "### Instruction:\nExplain LoRA in two sentences.\n\n### Response:\n"
    if tuned_model is None:
        raise RuntimeError("No tuned adapter loaded.")
    print(generate_text(tuned_model, tokenizer, prompt))
except Exception as error:
    print("Generation skipped. Load a base model and adapter first.")
    print(f"Reason: {error}")

## 7. Merge Adapters

Merging folds adapter weights into the base model. This can simplify deployment, but there are trade-offs.

Merge when:

- you only need one adapter,
- the serving stack expects a normal model,
- you want simpler artifact loading.

Do not merge when:

- you want to swap adapters dynamically,
- you need to keep many domain adapters,
- you are still experimenting and comparing adapters.

In [ ]:
merged_dir = Path("finetuning_outputs/merged_model")

try:
    if tuned_model is None:
        raise RuntimeError("No tuned adapter loaded.")
    merged_model = tuned_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f"saved merged model to: {merged_dir.resolve()}")
except Exception as error:
    print("Merge skipped. Load a PEFT adapter first.")
    print(f"Reason: {error}")

## 8. Serving Options

| Artifact | Serving path | Notes |
|---|---|---|
| Base model + adapter | Transformers or PEFT-aware serving | Flexible adapter swapping |
| Merged model | vLLM or Transformers | Simpler runtime load path |
| Quantized merged model | vLLM / specialized runtimes | Lower memory, extra compatibility checks |

For vLLM, serving LoRA adapters directly is possible in some configurations, but the simplest learning path is to merge the adapter and serve the merged model directory. Always confirm compatibility for your vLLM version and model family.

In [ ]:
vllm_command = f"""
vllm serve {merged_dir} \
  --host 0.0.0.0 \
  --port 8000 \
  --dtype auto \
  --max-model-len 2048
""".strip()

print(vllm_command)

## 9. Final Fine-Tuning Report Template

Every fine-tuning project should end with a short report. Use this structure:

```md
# Fine-Tuning Report

## Goal

## Base Model

## Dataset

## Formatting and Loss Masking

## Fine-Tuning Method

## Training Configuration

## Evaluation Results

## Serving Plan

## Known Regressions or Risks

## Next Improvements
```

## 10. Pass Criteria

A Week 5-6 fine-tuning deliverable is complete when:

- the dataset schema is documented,
- chat template or prompt format is documented,
- loss masking choice is documented,
- LoRA or QLoRA config is saved,
- adapter artifacts are saved,
- base and tuned outputs are compared,
- at least one automatic metric or rubric is reported,
- serving artifact choice is explained,
- known limitations are written down.

## Summary

Fine-tuning is a loop, not a single training command. You prepare data, format examples, train adapters, evaluate behavior, compare against the base model, save artifacts, decide whether to merge, and plan serving.

This completes the core Week 5-6 fine-tuning path. Preference tuning is a natural advanced extension after this.